# 01 — Data Acquisition (MIMIC-IV Demo)

This notebook downloads the open-access **MIMIC-IV Clinical Database Demo (v2.2)**, loads its relational tables, and reshapes them into the **canonical hourly time-series** used by the rest of the pipeline.

> Replaces the former PhysioNet-2019 flat-file loader. The 2019 Challenge set has no urine output, which makes full KDIGO AKI staging impossible. The MIMIC-IV demo has `icu/outputevents`, so we can build the complete creatinine **and** urine-output criteria downstream.

The demo is **de-identified and open-access** — no PhysioNet credentialing is required (that gate applies only to the full MIMIC-IV).

In [ ]:
!pip install -q pandas numpy pyarrow tqdm scipy scikit-learn xgboost lightgbm shap matplotlib seaborn imbalanced-learn optuna boto3

## Download the MIMIC-IV Demo

The demo is mirrored on the **PhysioNet open-data S3 bucket** (`physionet-open/mimic-iv-demo/`) and is downloaded anonymously — no AWS account or PhysioNet login needed. The cell below is idempotent: it skips any file already present, so it is safe to re-run.

The data lands in `../data/raw/mimic_iv_demo/`, preserving the `hosp/` and `icu/` folder structure that `data_loader` expects.

In [ ]:
import os

try:
    import boto3
except ImportError:
    !pip install -q boto3
    import boto3
from botocore import UNSIGNED
from botocore.config import Config

dest = "../data/raw/mimic_iv_demo"
os.makedirs(dest, exist_ok=True)

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
bucket, prefix = "physionet-open", "mimic-iv-demo/"

paginator = s3.get_paginator("list_objects_v2")
n_new = 0
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith("/"):
            continue
        local = os.path.join(dest, os.path.relpath(key, prefix))
        os.makedirs(os.path.dirname(local), exist_ok=True)
        if not os.path.exists(local):
            s3.download_file(bucket, key, local)
            n_new += 1

print(f"✓ MIMIC-IV demo ready in {dest} ({n_new} new files downloaded).")

In [ ]:
import sys
from pathlib import Path

# Add the project root to sys.path so we can import from src
sys.path.append(str(Path.cwd().parent))

from src.data_loader import (
    load_mimic_demo,
    build_hourly_cohort,
    identify_sepsis_onset,
    print_data_summary,
)

data_dir = "../data/raw/mimic_iv_demo"
print(f"Data directory set to: {data_dir}")

print("Loading MIMIC-IV demo tables...")
tables = load_mimic_demo(data_dir)
for name, t in tables.items():
    print(f"  {name:20s} {len(t):>9,} rows")

In [ ]:
import pandas as pd

print("Building hourly time-series cohort (one row per ICU stay-hour)...")
hourly = build_hourly_cohort(tables)

print("\nIdentifying sepsis onset (Sepsis-3 suspicion-of-infection proxy)...")
hourly = identify_sepsis_onset(hourly, tables)

print()
print_data_summary(hourly)
display(hourly.head())

# Persist the raw hourly frame for the cohort-definition notebook (02)
interim = Path("../data/interim")
interim.mkdir(parents=True, exist_ok=True)
hourly.to_parquet(interim / "hourly_cohort.parquet", index=False)
print(f"\nSaved → {interim / 'hourly_cohort.parquet'}  (shape={hourly.shape})")

## Conclusion

The MIMIC-IV demo has been downloaded, its relational tables loaded, and reshaped into the canonical hourly time-series (`hourly_cohort.parquet`) — including the **urine-output rate and its observation mask**, which the downstream KDIGO labeling relies on.

**Next:** `02_cohort_definition.ipynb` applies inclusion criteria, computes baseline creatinine, and excludes prevalent AKI to produce the final modeling cohort.

> ⚠️ Sepsis onset currently uses the *suspicion-of-infection* half of Sepsis-3 (antibiotic + culture pairing). The SOFA ≥ 2 component is a documented simplification for the demo — see `identify_sepsis_onset` in `src/data_loader.py`.